In [ ]:
import numpy as np
import cvxpy as cp
import matplotlib.pyplot as plt

In [ ]:
# Problem data.
m = 1 # pendulum mass
g = 10 # gravity acceleration
l = 1 # rod length
h = .05 # discretization time step
K = 100 # number of time steps

# Final state.
xK = np.array([0, 0])

In [ ]:
# Discrete-time linearized dynamics.
A = np.eye(2) + h * np.vstack([[0, 1], [g / l, 0]])
B = h * cp.hstack([[0], [1 / (m * l ** 2)]])
def f_lin(x, u):
    return A @ x + B * u

In [ ]:
def optimal_control(x0):

    # Variables.
    X = cp.Variable((K, 2)) # matrix with the state at each time step
    U = cp.Variable(K - 1) # vector with the control input at each time step

    # Initial and final conditions.
    constraints = [X[0] == x0, X[-1] == xK]

    # Discrete-time linearized dynamics.
    for k in range(K - 1):
        constraints.append(X[k + 1] == f_lin(X[k], U[k])) 

    # Objective function.
    obj = h * cp.sum_squares(U)

    # Solve problem.
    prob = cp.Problem(cp.Minimize(obj), constraints)
    prob.solve()

    return U.value, X.value

In [ ]:
# State trajectory for linearized systems.
plt.figure()
initial_conditions = np.linspace([-np.pi, 0], [0, 0], 10)
for x0 in initial_conditions:
    U_lin, X_lin = optimal_control(x0)
    plt.plot(*X_lin.T)
plt.xlabel(r'Angle $q$')
plt.ylabel(r'Angular velocity $\dot q$')
plt.title('Linearize pendulum swing-up trajectory')
plt.grid()

In [ ]:
# Discrete-time dynamics.
def f(x, u):
    dq = x[1]
    ddq = g * np.sin(x[0]) / l + u / (m * l ** 2)
    return x + h * np.array([dq, ddq])

# Simulate nonlinear system given initial state and control inputs.
def simulate(x0, U):
    X = np.zeros((K, 2))
    X[0] = x0
    for k in range(K - 1):
        X[k + 1] = f(X[k], U[k])
    return X

# State trajectory for nonlinear systems.
plt.figure()
for x0 in initial_conditions:
    U_lin = optimal_control(x0)[0]
    X = simulate(x0, U_lin)
    plt.plot(*X.T)
plt.xlabel(r'Angle $q$')
plt.ylabel(r'Angular velocity $\dot q$')
plt.title('Actual trajectories of nonlinear pendulum')
plt.grid()